# C-11 · YOLO26n-OBB trên AI-TOD-R · các quy tắc vùng ứng viên A0/A1/A2/A8 (+A4)

Notebook này chạy **proposal đã nộp C-10/C-11** của Nhóm 7 (`notes/02-proposal/t07-c3-proposal-v2-EN.md`,
hộp v2.1 + đính chính C-19). Tài liệu giải thích: `notes/08-c11-execution/`.

**Luồng (chạy theo thứ tự, mỗi phần có thể chạy lại):**

| Phần | Việc | Cần GPU? | Cổng ROADMAP |
|---|---|---|---|
| 0 | Cấu hình, Drive, cài đúng phiên bản, chạy unit test | không | — |
| 1 | Lấy dữ liệu (nhãn + ảnh không thuộc xView) | không | 0, 1 |
| 2 | Cổng 0: thống kê nhãn | không | 0 |
| 3 | Cổng 2(b)(c): oracle + vòng tròn cho bộ chấm | không | 2 |
| 4 | **RQ2 offline**: ứng viên theo từng arm (chỉ nhãn) | không | — |
| 5 | Dựng dataset YOLO (lọc đúng luật loader gốc, tách val theo cảnh) | không | — |
| 6 | Pilot: đo ảnh/giây | **có** | 2(a) |
| 7 | Train arm × seed (tự resume khi Colab ngắt) | **có** | — |
| 8 | Predict + chấm bằng AITODReval gốc (2 head) | **có** | — |
| 9 | Gộp kết quả | không | — |

**Quy tắc (AGENTS.md §6):** không gõ số tay vào báo cáo — mọi số đọc từ file JSON mà notebook ghi ra
`experiments/`. Chưa có preregistration (có ngày) thì **chưa** chạy phần 7 cho so sánh chính.

**Phần 1–5 chạy được trên runtime CPU** (rẻ, không tốn đơn vị GPU). Chỉ đổi sang GPU cho phần 6–8.

## 0. Cấu hình

Sửa các đường dẫn cho đúng Drive của nhóm. `PROTOCOL`:
- `"woX"` — **chỉ ảnh không thuộc xView** (ROADMAP Cổng 1 fallback). Bỏ ảnh xView ở **cả** train lẫn test.
  Kết quả **không so với Bảng II** của bài AI-TOD-R; ghi rõ trong bài.
- `"full"` — đủ 28.036 ảnh (cần xView). Chỉ dùng khi đã dựng đủ ảnh.

In [39]:
from pathlib import Path
import os, json, time, sys

DRIVE      = Path("/content/drive/MyDrive/NCKH")          # thư mục nhóm trên Drive (sửa)
CODE_DIR   = DRIVE / "repo/NCKH/scripts/c11"              # bản sao scripts/c11 của repo
DATA_DRIVE = DRIVE / "data"                               # nơi để labels.zip và các zip ảnh
EXP_DRIVE  = DRIVE / "repo/NCKH/experiments"              # kết quả ghi về đây (đồng bộ lại repo sau)
LOCAL      = Path("/content/aitodr")                      # đĩa local của VM: giải nén + train ở đây (nhanh)

PROTOCOL   = "woX"           # "woX" hoặc "full"
VERSION    = "oc"         # angle version cho loader/evaluator (config DCFL của AI-TOD-R dùng le135)
ARMS       = ["A0", "A1", "A8", "A2"]   # so sánh chính (C-11); "A4" là phép thử phụ
SEEDS      = [1, 2, 3]
DIAG_EVERY = 10              # ghi chẩn đoán gán nhãn 1/10 lần gọi assigner (≈ +2–3% thời gian; 0 = tắt)
TODAY      = time.strftime("%Y-%m-%d")
print(TODAY, PROTOCOL, ARMS, SEEDS)

2026-10-20 woX ['A0', 'A1', 'A8', 'A2'] [1, 2, 3]


In [40]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "CPU runtime"
!df -h /content | tail -1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
overlay         236G   74G  163G  32% /


### 0.1 Cài đúng phiên bản

- `ultralytics==8.4.174`: **khoá cứng**. Bản vá `stal_arms.py` kiểm phiên bản và từ chối chạy nếu khác,
  vì quy tắc STAL cho OBB đã đổi 3 lần trong 2026 (proposal §2.2).
- `shapely`: tính IoU hộp xoay cho AITODReval khi không cài được `mmcv` (sai khác P2, xem `aitodr_evalkit.py`).
- `mmcv` **không bắt buộc**. Nếu muốn dùng IoU gốc của mmcv, cài trong môi trường riêng rồi chạy
  `check_iou_backends()` để so với shapely.

In [41]:
!pip -q install "ultralytics==8.4.174" shapely gdown
import ultralytics, torch, numpy, platform
ENV = dict(ultralytics=ultralytics.__version__, torch=torch.__version__, numpy=numpy.__version__,
           python=platform.python_version(),
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print(ENV)
assert ENV["ultralytics"] == "8.4.174"

{'ultralytics': '8.4.174', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'python': '3.13.15', 'gpu': 'NVIDIA A100-SXM4-40GB'}


### 0.2 Lấy code và chạy unit test

44 test (đã qua trên máy ngày 20/10/2026, CPU): STAL arms (A2 trùng mã gốc, A0 khớp phép thử độc lập,
tập ứng viên lồng nhau, tính chất P1–P5 của OA-STAL v2, Mệnh đề 1), bộ chấm (oracle, vòng tròn,
đặc tính id-0 của evaluator gốc), dữ liệu và RQ2 offline. **Phải qua hết trên Colab** trước khi làm tiếp.

In [42]:
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
sys.path.insert(0, "/content/c11")
!cd /content/c11 && python -m pytest -q test_stal_arms.py test_evalkit.py test_data_rq2.py test_bootstrap.py 2>&1 | tail -3

............................................                             [100%]
44 passed in 20.19s


## 1. Dữ liệu

**Nhãn:** `labels.zip` (16 MB) từ link chính thức trên trang AI-TOD-R (Google Drive id
`1gFaoc5p5whoeDsnsLt6F18O6FHbLGtlK`). Ngày 20/10 link trả "Quota exceeded" (hết hạn mức tải công khai).
Cách hợp lệ: một thành viên đăng nhập Google, mở link, **Make a copy** vào `DATA_DRIVE`, rồi notebook đọc
bản copy. Không tìm cách lách hạn mức.

**Ảnh không thuộc xView:** thư mục Drive `AI-TOD_wo_xview` (id `1uNY_rcOO5LrWibXRY6l2dvqSbK6xikJp`), cần
`aitod_wo_xview_trainval_imgs.zip` (7.4G) và `aitod_wo_xview_test_imgs.zip` (7.4G). Tương tự: Make a copy
vào `DATA_DRIVE`. Cần Drive trống ≥ 15 GB (gói Drive miễn phí 15 GB là không đủ).

**Ảnh xView** (chỉ cho `PROTOCOL="full"`): một thành viên tự đăng ký xView, rồi chạy toolkit AI-TOD.
Xem hợp đồng dữ liệu §3–4.

In [43]:
LOCAL.mkdir(parents=True, exist_ok=True)
lab_zip = DATA_DRIVE / "labels.zip"
if not lab_zip.exists():
    # thử tải trực tiếp; nếu Drive báo quota, file tải về sẽ là HTML -> báo lỗi rõ ràng
    !gdown "1gFaoc5p5whoeDsnsLt6F18O6FHbLGtlK" -O "{lab_zip}" || true
assert lab_zip.exists() and open(lab_zip, "rb").read(2) == b"PK", "labels.zip chưa có hoặc là trang HTML (quota). Make a copy vào DATA_DRIVE."
!unzip -q -o "{lab_zip}" -d "{LOCAL}"
!find "{LOCAL}" -maxdepth 3 -type d | head -20

/content/aitodr
/content/aitodr/labels
/content/aitodr/labels/test
/content/aitodr/labels/trainval
/content/aitodr/images_raw
/content/aitodr/images_raw/images
/content/aitodr/yolo_woX
/content/aitodr/yolo_woX/images
/content/aitodr/yolo_woX/images/test
/content/aitodr/yolo_woX/images/val
/content/aitodr/yolo_woX/images/train
/content/aitodr/yolo_woX/labels
/content/aitodr/yolo_woX/labels/test
/content/aitodr/yolo_woX/labels/val
/content/aitodr/yolo_woX/labels/train


In [44]:
# Sửa nếu cấu trúc trong zip khác (xem output ô trên)
LABELS = next(p.parent for p in LOCAL.rglob("trainval") if p.is_dir() and any(p.glob("*.txt")))
print("LABELS =", LABELS, [d.name for d in LABELS.iterdir()])

IMAGES = LOCAL / "images_raw"
IMAGES.mkdir(exist_ok=True)
for z in ["aitod_wo_xview_trainval_imgs.zip", "aitod_wo_xview_test_imgs.zip"]:
    src = DATA_DRIVE / z
    if src.exists() and not (IMAGES / (z + ".done")).exists():
        !cp "{src}" /content/ && unzip -q -o "/content/{z}" -d "{IMAGES}" && rm "/content/{z}" && touch "{IMAGES}/{z}.done"
    print(z, "OK" if (IMAGES / (z + ".done")).exists() else "CHƯA CÓ")
print("số PNG:", sum(1 for _ in IMAGES.rglob("*.png")))

LABELS = /content/aitodr/labels ['test', 'trainval']
aitod_wo_xview_trainval_imgs.zip OK
aitod_wo_xview_test_imgs.zip OK
số PNG: 18730


## 2. Cổng 0 — thống kê nhãn (không cần ảnh)

Trả lời các câu (a)–(d) của ROADMAP Cổng 0. Đặc biệt:
- `n_image_class_pairs_over_1500`: số (ảnh, lớp) có hơn 1.500 GT. AITODReval chỉ giữ 1.500 dự đoán cho
  mỗi (ảnh, lớp), nên recall ở các cặp này bị chặn **như nhau cho mọi phương pháp**.
- `short_side_lt_8 / lt_11p31 / lt_16`: số GT mà A1 / A8 / A2 thực sự thay đổi. Đây là "tỉ lệ bị ảnh hưởng"
  dùng cho H4.
- `n_xview_files` và `n_non_xview_files`: số chính xác của giao thức woX.

In [45]:
import aitodr_data as ad
OUT = EXP_DRIVE / f"{TODAY}-gate0"
OUT.mkdir(parents=True, exist_ok=True)
g0 = ad.gate0_stats(LABELS, version=VERSION)
(OUT / "gate0_stats.json").write_text(json.dumps(g0, indent=2, ensure_ascii=False))
print(json.dumps(g0, indent=1, ensure_ascii=False)[:4000])
XVIEW = ad.xview_stems()
EXCLUDE = (XVIEW["trainval"] | XVIEW["test"]) if PROTOCOL == "woX" else set()

{
 "version": "oc",
 "trainval": {
  "n_files": 14018,
  "gt_raw": 380991,
  "gt_kept": 380174,
  "gt_dropped_lt2px": 817,
  "files_all_gt_dropped": 8,
  "empty_files": 17,
  "n_images_evaluated": 14001,
  "n_xview_files": 4639,
  "n_xview_listed": 4639,
  "n_xview_missing_in_labels": 0,
  "n_non_xview_files": 9379,
  "classes": {
   "person": 17886,
   "vehicle": 333744,
   "airplane": 842,
   "ship": 17597,
   "storage-tank": 8104,
   "bridge": 783,
   "swimming-pool": 936,
   "wind-mill": 282
  },
  "area_bins": {
   "small": 23216,
   "tiny": 262047,
   "medium": 3938,
   "very_tiny": 90954,
   "above_64px": 19
  },
  "max_gt_per_image": 1435,
  "max_gt_per_image_class": 1435,
  "n_image_class_pairs_over_1500": 0,
  "aspect_quantiles": {
   "p50": 2.0,
   "p90": 3.008,
   "p99": 4.981
  },
  "short_side_lt_8": 252854,
  "short_side_lt_11p31": 352402,
  "short_side_lt_16": 367929,
  "long_side_lt_16": 264635,
  "n_scene_groups": 8775
 },
 "test": {
  "n_files": 14018,
  "gt_raw": 37

## 3. Cổng 2(b)(c) — bộ chấm có đáng tin không

- **Vòng tròn (c):** đa giác → (x, y, w, h, a) → đa giác. IoU phải ≈ 1. Đa giác gốc không phải chữ nhật
  hoàn hảo thì IoU < 1 một chút: đó là đặc tính của dữ liệu và loader gốc. Ghi lại, không "sửa".
- **Oracle (b):** đưa chính GT vào làm dự đoán. AP ≈ 1 (thang 0–1). Có hai chỗ AP **không** bằng đúng 1:
  1. các (ảnh, lớp) có > 1.500 GT;
  2. **đặc tính của AITODReval gốc**: id của GT và dự đoán bắt đầu từ 0, nhưng cờ "đã khớp" lại là id.
     Vì vậy cặp có id 0 luôn bị tính là chưa khớp (1 FP + 1 FN). Ảnh hưởng ~1/số GT của một lớp, như nhau
     cho mọi phương pháp. **Giữ nguyên, không sửa mã gốc.**

In [46]:
import aitodr_evalkit as ek
OUT = EXP_DRIVE / f"{TODAY}-gate2-evaluator"
OUT.mkdir(parents=True, exist_ok=True)
rt = ek.roundtrip_check(LABELS / "test", VERSION)
infos = ek.load_gt(LABELS / "test", VERSION, exclude_stems=EXCLUDE)
import numpy as np
preds = {i["stem"]: (np.array([ek.obb2poly_np(*b) for b in i["bboxes"]]).reshape(-1, 8), i["labels"],
                     np.ones(len(i["labels"]))) for i in infos}
oracle = ek.evaluate(infos, ek.predictions_to_results(preds, infos, VERSION), verbose=True)
rep = dict(roundtrip=rt, oracle=oracle, protocol=PROTOCOL, n_images=len(infos), env=ENV,
           iou_backend_diff_vs_mmcv=ek.check_iou_backends())
(OUT / "gate2_evaluator.json").write_text(json.dumps(rep, indent=2))
print(json.dumps(rep, indent=1)[:2000])

Evaluate annotation type *mAP*
Calculating IoUs...
IoU calculation Done (t=15.83s).
Running per image evaluation...
DONE (t=470.68s).
Accumulating evaluation results...
DONE (t=1.16s).
Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=1500 ] = 0.996
Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1500 ] = 0.996
Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=1500 ] = 0.996
Average Precision  (AP) @[ IoU=0.50:0.95 | area=very tiny | maxDets=1500 ] = 0.998
Average Precision  (AP) @[ IoU=0.50:0.95 | area=  tiny | maxDets=1500 ] = 0.996
Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=1500 ] = 1.000
Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=1500 ] = 1.000
Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=1500 ] = 0.998
Average Recall     (AR) @[ IoU=0.50:0.95 | area=very tiny | maxDets=1500 ] = 0.999
Average Recall     (AR) @[ IoU=0.50:0.95 | area=  tiny | maxDets=1500 ] = 0.998
Average R

## 4. RQ2 offline — trung tâm của bài (chỉ cần nhãn)

Với **mỗi GT** của trainval và **mỗi arm**, đếm:

- số ứng viên;
- có rỗng không;
- số ứng viên dùng chung với GT khác (nguồn xung đột);
- số ứng viên nằm ngoài vật;
- số ứng viên theo từng tầng stride.

Phép thử điểm-trong-hộp gọi đúng hàm của Ultralytics 8.4.174, nên khớp với lúc train.

**Đọc kết quả cho đúng (C-19):**

- Đây là tầng **ứng viên**, chưa phải positive cuối cùng. Positive còn qua top-k và xử lý xung đột; phần
  đó đo online ở phần 7 (`c11_assign_diag.jsonl`).
- Có ứng viên ≠ có positive ≠ giám sát hữu ích.
- Bảng này dùng để **khoá dấu dự báo trong preregistration TRƯỚC khi train**.

Chạy trên toàn bộ trainval có thể mất khá lâu trên CPU. Đặt `MAX_FILES` nhỏ để thử trước.

In [47]:
import rq2_offline as rq
MAX_FILES = 300          # None = toàn bộ; thử nhỏ trước
OUT = EXP_DRIVE / f"{TODAY}-rq2-offline-{PROTOCOL}"
summ = rq.analyze(LABELS / "trainval", OUT, arms=("A0", "A1", "A8", "A2", "A4"), version=VERSION,
                  exclude_stems=EXCLUDE, max_files=MAX_FILES)
import pandas as pd
rows = [dict(cell=k, arm=a, **v) for k, per in summ["size_ar"].items() for a, v in per.items()]
df = pd.DataFrame(rows)
display(df.pivot_table(index="cell", columns="arm", values="zero_cand_rate").round(3))
display(df.pivot_table(index="cell", columns="arm", values="shared_frac").round(3))

[rq2] 0/300 file, 1 GT


arm,A0,A1,A2,A4,A8
cell,,,,,
l>=64|r2-4,0.000,0.0,0.0,0.0,0.0
l>=64|r<2,0.000,0.0,0.0,0.0,0.0
m32-64|r2-4,0.000,0.0,0.0,0.0,0.0
m32-64|r<2,0.000,0.0,0.0,0.0,0.0
m32-64|r>=4,0.000,0.0,0.0,0.0,0.0
s16-32|r2-4,0.000,0.0,0.0,0.0,0.0
s16-32|r<2,0.000,0.0,0.0,0.0,0.0
s16-32|r>=4,0.000,0.0,0.0,0.0,0.0
t8-16|r2-4,0.024,0.0,0.0,0.0,0.0


arm,A0,A1,A2,A4,A8
cell,,,,,
l>=64|r2-4,0.000,0.000,0.000,0.000,0.000
l>=64|r<2,0.000,0.000,0.000,0.000,0.000
m32-64|r2-4,0.108,0.110,0.113,0.111,0.109
m32-64|r<2,0.050,0.051,0.055,0.052,0.051
m32-64|r>=4,0.000,0.000,0.000,0.000,0.000
s16-32|r2-4,0.222,0.228,0.281,0.231,0.232
s16-32|r<2,0.118,0.124,0.139,0.131,0.122
s16-32|r>=4,0.214,0.207,0.317,0.214,0.271
t8-16|r2-4,0.205,0.307,0.370,0.287,0.273


In [48]:
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
!cd /content/c11 && python -m pytest -q test_stal_arms.py test_evalkit.py test_data_rq2.py test_bootstrap.py 2>&1 | tail -1
import importlib, aitodr_data as ad
importlib.reload(ad)

44 passed in 19.95s


<module 'aitodr_data' from '/content/c11/aitodr_data.py'>

## 5. Dựng dataset YOLO

- Lọc **đúng luật loader gốc**: bỏ file nhãn rỗng (cả ảnh), bỏ GT có cạnh < 2 px. Nhờ vậy tập YOLO học khớp
  với tập GT mà AITODReval chấm.
- Val = 10% trainval, **tách theo nhóm cảnh**: ảnh xView nhóm theo tile gốc; quy ước tên của phần không phải
  xView chưa xác minh. Mọi tinh chỉnh chỉ làm trên val; test chỉ chạm một lần cho mỗi arm cuối cùng
  (proposal §8.1).
- Ảnh được symlink, không copy.

In [49]:
DS = LOCAL / f"yolo_{PROTOCOL}"
man = ad.build_yolo_dataset(LABELS, IMAGES, DS, version=VERSION, exclude_xview=(PROTOCOL == "woX"),
                            val_frac=0.10, seed=0)
print(json.dumps({k: man[k] for k in ("splits", "missing_images", "dropped_lt2px", "n_gt_clipped", "max_overflow_px",
                                     "n_images_all_gt_dropped", "n_images_not_img_size")}, indent=1))
if man["n_images_not_img_size"]:
    for f in DS.glob("odd_size_*.txt"):
        print(f.name, f.read_text().splitlines())
(EXP_DRIVE / f"{TODAY}-dataset-{PROTOCOL}").mkdir(parents=True, exist_ok=True)
(EXP_DRIVE / f"{TODAY}-dataset-{PROTOCOL}" / "split_manifest.json").write_text(json.dumps(man, indent=2))
assert sum(man["missing_images"].values()) == 0, "thiếu ảnh — xem missing_*.txt"
DATA_YAML = DS / "aitodr.yaml"

{
 "splits": {
  "train": {
   "n_images": 8426,
   "n_gt": 123272,
   "n_scene_groups": 7326
  },
  "val": {
   "n_images": 936,
   "n_gt": 12003,
   "n_scene_groups": 815
  },
  "test": {
   "n_images": 9261,
   "n_gt": 128768,
   "n_scene_groups": 8110
  }
 },
 "missing_images": {
  "train": 0,
  "val": 0,
  "test": 0
 },
 "dropped_lt2px": 1345,
 "n_gt_clipped": 2188,
 "max_overflow_px": 54.0,
 "n_images_all_gt_dropped": {
  "train": 8,
  "val": 0,
  "test": 3
 },
 "n_images_not_img_size": 7
}
odd_size_test.txt ['01076 800x787', '15324 800x787', '16819 800x797', '20730 800x782']
odd_size_val.txt ['12490 800x788']
odd_size_train.txt ['12286 800x801', '08561 800x799']


## 6. Pilot — đo tốc độ (Cổng 2a) · cần GPU

Train 1 epoch arm A2 để đo ảnh/giây và GPU-giờ cho mỗi run. Từ đó tính lại ngân sách:
GPU-giờ/run = N_ảnh × epochs / (ảnh/giây × 3600) (proposal §9).

In [50]:
import c11_pipeline as cp
t0 = time.time()
run = cp.train_arm(DATA_YAML, "A2", seed=0, project=LOCAL / "runs", tag="pilot", diag_every=DIAG_EVERY,
                   recipe_overrides=dict(epochs=1))
dt = time.time() - t0
n_train = man["splits"]["train"]["n_images"]
pilot = dict(seconds=dt, n_train=n_train, img_per_s=n_train / dt, est_gpu_h_per_run=24 * n_train / (n_train / dt) / 3600,
             env=ENV, recipe=cp.RECIPE)
(EXP_DRIVE / f"{TODAY}-pilot").mkdir(parents=True, exist_ok=True)
(EXP_DRIVE / f"{TODAY}-pilot" / "pilot.json").write_text(json.dumps(pilot, indent=2, default=str))
print(pilot)

Transferred 708/792 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/aitodr/yolo_woX/aitodr.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio

## 7. Train arm × seed · cần GPU

**Trước khi chạy cho so sánh chính:** commit `experiments/preregistration.md` có ngày (so sánh chính
A0/A1/A2/A8, dấu dự báo theo mật độ lấy từ phần 4, hiệu ứng tối thiểu, code phân tích). Không sửa dự báo
sau khi đã thấy kết quả.

**Cách chạy an toàn trên Colab:**
- Thư mục run nằm trên **Drive** (`RUNS`), `save_period=1`.
- Colab ngắt thì chạy lại ô này: run nào có `last.pt` mà chưa có `c11_done.json` sẽ **resume**.
- Mỗi run ghi:
  - `c11_stal_arm.json`: arm, phiên bản, sha256 hàm gốc, số lần hàm vá được gọi;
  - `c11_assign_diag.jsonl`: theo epoch × nhánh o2m/o2o × ô kích thước/tỉ lệ cạnh: ứng viên, positive
    trước/sau xung đột, tỉ lệ GT 0 positive, positive "nhận thêm" qua xung đột.
- Callback **báo lỗi** nếu hàm vá không được gọi trong một epoch, tức là vá không có hiệu lực.

In [51]:
RUNS = EXP_DRIVE / f"runs-{PROTOCOL}"      # trên Drive để sống sót khi Colab ngắt
QUEUE = [(a, s) for s in SEEDS for a in ARMS]  # xen kẽ arm theo seed: cặp so sánh chạy gần nhau, cùng loại GPU
for arm, seed in QUEUE:
    rd = RUNS / cp.run_name(arm, seed)
    if (rd / "c11_done.json").exists():
        print("đã xong", rd.name); continue
    print(">>> train", arm, seed, "GPU:", ENV["gpu"])
    cp.train_arm(DATA_YAML, arm, seed, project=RUNS, diag_every=DIAG_EVERY)

>>> train A0 1 GPU: NVIDIA A100-SXM4-40GB
Transferred 708/792 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/aitodr/yolo_woX/aitodr.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=24, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=800, iou=0.7, kobj=1.0, line

## 8. Predict + chấm bằng AITODReval gốc

- `max_det=12000` (1.500 × 8 lớp, vì evaluator cắt theo (ảnh, lớp)), `conf=0.001`.
- Hai head của YOLO26 báo **riêng** (proposal §8.2):
  - `e2e`: one-to-one, không NMS (`nms=False`);
  - `nms`: one-to-many + NMS (mặc định của Ultralytics 8.4.174 khi `nms=None`).
- Chấm `last.pt`, không chọn `best.pt` (đã khai báo trước trong `c11_pipeline.py`).
- File dự đoán dạng DOTA Task1 được lưu để phát hành (`dota_<head>/`).

In [66]:
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
!cd /content/c11 && python -m pytest -q test_stal_arms.py test_evalkit.py test_data_rq2.py test_bootstrap.py test_fastpath.py 2>&1 | tail -1
import importlib, aitodr_evalkit as ek, c11_bootstrap as cb, c11_pipeline as cp
for m in (ek, cb, cp): importlib.reload(m)
print(ek.load_official_evaluator()[1])

55 passed in 69.93s (0:01:09)
shapely+P3sparse


In [63]:
!ls -la "{CODE_DIR}" | grep -E "\.py$"
!head -3 "{CODE_DIR}/diag_bootstrap_device.py"

-rw------- 1 root root 11225 Oct 20 13:55 aitodr_data.py
-rw------- 1 root root 26516 Oct 21 02:55 aitodr_evalkit.py
-rw------- 1 root root 20201 Oct 20 11:31 build_notebook.py
-rw------- 1 root root 20902 Oct 21 06:10 c11_bootstrap.py
-rw------- 1 root root  7890 Oct 21 02:56 c11_pipeline.py
-rw------- 1 root root  2625 Oct 20 06:57 check_geometry_c11.py
-rw------- 1 root root  3614 Oct 21 06:16 diag_bootstrap_device.py
-rw------- 1 root root  8716 Oct 20 08:29 rq2_offline.py
-rw------- 1 root root 24286 Oct 20 07:08 stal_arms.py
-rw------- 1 root root  6719 Oct 21 06:10 test_bootstrap.py
-rw------- 1 root root  6065 Oct 20 13:56 test_data_rq2.py
-rw------- 1 root root  5665 Oct 21 02:56 test_evalkit.py
-rw------- 1 root root  7004 Oct 21 02:46 test_fastpath.py
-rw------- 1 root root 15359 Oct 20 07:08 test_stal_arms.py
"""Chẩn đoán: engine "weighted" trên GPU có cho cùng số với bản tham chiếu (CPU, numpy) không.

Chạy trên Colab:  !cd /content/c11 && python diag_bootstrap_device.py


In [65]:
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
!cd /content/c11 && python diag_bootstrap_device.py 2>&1 | grep -v "^Evaluate\|^Calc\|^IoU\|^Running\|^DONE\|^Accum\|Average"

torch 2.11.0+cu130 | cuda True NVIDIA A100-SXM4-40GB
tổng số mẫu lệch: 0/200


In [61]:
!cd /content/c11 && python -m pytest -q test_stal_arms.py test_evalkit.py test_data_rq2.py test_bootstrap.py test_fastpath.py 2>&1 | grep -E "FAILED|Error" | head -5

test_bootstrap.py:109: AssertionError
FAILED test_bootstrap.py::test_multi_pair_point_equals_official - assert 0.02...


In [54]:
ref_dir = next(rd for rd in sorted(RUNS.glob("c11_*_s*")) if (rd / "eval_e2e.json").exists())
old = json.loads((ref_dir / "eval_e2e.json").read_text())
t0 = time.time()
new = cp.predict_and_eval(ref_dir / "weights" / "last.pt", LABELS / "test", DS / "images" / "test",
                          ref_dir / "eval_e2e_recheck.json", head="e2e",
                          version=old.get("version", VERSION), exclude_stems=EXCLUDE)
diff = {k: abs(old[k] - new[k]) for k in ek.STAT_NAMES}
print(ref_dir.name, "bản mới mất", round(time.time() - t0), "giây")
print("chênh lệch lớn nhất:", max(diff.values()))

c11_A0_s1 bản mới mất 337 giây
chênh lệch lớn nhất: 9.428956928361765e-07


In [56]:
TEST_IMAGES = DS / "images" / "test"
for rd in sorted(RUNS.glob("c11_*_s*")):
    if not (rd / "c11_done.json").exists():
        continue
    for head in ("e2e",):
        out = rd / f"eval_{head}.json"
        if out.exists():
            continue
        st = cp.predict_and_eval(rd / "weights" / "last.pt", LABELS / "test", TEST_IMAGES, out, head=head,
                                 version=VERSION, exclude_stems=EXCLUDE, save_dota_dir=rd / f"dota_{head}")
        print(rd.name, head, {k: round(st[k], 4) for k in ("AP", "AP50", "AP_vt", "AP_t")})

c11_A1_s3 e2e {'AP': 0.1293, 'AP50': 0.3907, 'AP_vt': 0.0504, 'AP_t': 0.1183}
c11_A2_s1 e2e {'AP': 0.1321, 'AP50': 0.3971, 'AP_vt': 0.0456, 'AP_t': 0.1242}
c11_A2_s2 e2e {'AP': 0.1384, 'AP50': 0.4132, 'AP_vt': 0.057, 'AP_t': 0.1275}
c11_A2_s3 e2e {'AP': 0.1292, 'AP50': 0.3966, 'AP_vt': 0.0472, 'AP_t': 0.1192}
c11_A8_s1 e2e {'AP': 0.1304, 'AP50': 0.3924, 'AP_vt': 0.0453, 'AP_t': 0.1198}
c11_A8_s2 e2e {'AP': 0.1302, 'AP50': 0.3936, 'AP_vt': 0.047, 'AP_t': 0.1213}
c11_A8_s3 e2e {'AP': 0.1359, 'AP50': 0.4112, 'AP_vt': 0.0517, 'AP_t': 0.1238}


## 9. Gộp kết quả

Bảng mean ± sd theo arm và head, đọc thẳng từ các file `eval_*.json`. Đây là **mô tả**, chưa phải kết luận:

- So sánh chính cần **bootstrap theo cụm cảnh** (proposal §8.6); code phân tích nằm trong preregistration.
- Gọi một khác biệt là "cải thiện" chỉ khi CI không chứa 0 **và** mẫu hình dự báo trước đúng.

In [57]:
import pandas as pd
recs = []
for f in RUNS.glob("c11_*_s*/eval_*.json"):
    d = json.loads(f.read_text())
    done = json.loads((f.parent / "c11_done.json").read_text())  # arm/seed lấy từ file, không parse tên thư mục
    recs.append(dict(arm=done["arm"], seed=done["seed"], head=d["head"], batch=done.get("batch_actual"),
                     batch_changed=done.get("batch_changed"), **{k: d[k] for k in ek.STAT_NAMES}))
res = pd.DataFrame(recs)
if res["batch_changed"].any():
    print("CẢNH BÁO: có run bị Ultralytics đổi batch (OOM) -> các arm không cùng recipe:", res[res.batch_changed])
agg = res.groupby(["head", "arm"])[["AP", "AP50", "AP_vt", "AP_t", "AP_s", "AR1500"]].agg(["mean", "std", "count"])
display(agg.round(4))
res.to_csv(EXP_DRIVE / f"{TODAY}-results-table-{PROTOCOL}.csv", index=False)

AP                  AP50                 AP_vt                \
            mean     std count    mean     std count    mean     std count   
head arm                                                                     
e2e  A0   0.1301  0.0016     4  0.3882  0.0035     4  0.0410  0.0006     4   
     A1   0.1303  0.0024     3  0.3925  0.0103     3  0.0484  0.0038     3   
     A2   0.1332  0.0047     3  0.4023  0.0094     3  0.0499  0.0062     3   
     A8   0.1322  0.0032     3  0.3991  0.0105     3  0.0480  0.0033     3   
nms  A0   0.1356  0.0031     3  0.3994  0.0094     3  0.0412  0.0026     3   
     A1   0.1350     NaN     1  0.3979     NaN     1  0.0446     NaN     1   

            AP_t                  AP_s                AR1500                
            mean     std count    mean     std count    mean     std count  
head arm                                                                    
e2e  A0   0.1173  0.0021     4  0.2092  0.0056     4  0.2769  0.0018     4  
     A1   0.1190  0.0035     3  0.2014  0.0014     3  0.2774  0.0061     3  
     A2   0.1237  0.0042     3  0.2035  0.0084     3  0.2818  0.0050     3  
     A8   0.1216  0.0020     3  0.2051  0.0063     3  0.2787  0.0045     3  
nms  A0   0.1213  0.0034     3  0.2126  0.0068     3  0.2378  0.0033     3  
     A1   0.1203     NaN     1  0.2062     NaN     1  0.2396     NaN     1

### 9.1 So sánh chính — bootstrap ghép cặp theo cụm cảnh

Chạy **sau khi** đủ seed cho cả hai arm của mỗi cặp, và đúng các cặp đã ghi trong preregistration.
- Mỗi run có `evalcache_<head>.npz`: kết quả khớp từng (ảnh, lớp, bin) do AITODReval gốc tính.
  Bootstrap tính lại AP từ đó; đã kiểm khớp số chính thức tới 1e-12.
- Đơn vị lấy mẫu là **cụm cảnh** (`scene_key`), cùng một mẫu cho mọi run. Δ = trung bình theo seed arm X trừ arm Y.
- "Khác biệt" chỉ khi CI 95% không chứa 0 **và** |Δ| ≥ MEI đã chốt trong preregistration.

In [68]:
import c11_bootstrap as cb, torch
from aitodr_data import scene_key
HEAD  = "e2e"
PAIRS = [("A8", "A0"), ("A2", "A0"), ("A2", "A8"), ("A1", "A0")]
B     = 1000
KEYS  = ("AP_vt", "AP_t", "AP")
by_arm = {}
for rd in sorted(RUNS.glob("c11_*_s*")):
    f = rd / f"evalcache_{HEAD}.npz"
    if (rd / "c11_done.json").exists() and f.exists():
        by_arm.setdefault(json.loads((rd / "c11_done.json").read_text())["arm"], []).append(cb.load_cache(f))
print({a: len(v) for a, v in by_arm.items()}, "| GPU:", torch.cuda.is_available())
pairs = [(x, y) for x, y in PAIRS if len(by_arm.get(x, [])) >= 2 and len(by_arm.get(y, [])) >= 2]
t0 = time.time()
boot = cb.multi_pair_bootstrap({a: by_arm[a] for a in {z for p in pairs for z in p}}, pairs, scene_key,
                               keys=KEYS, B=B, engine="weighted")
print(f"xong {time.time() - t0:.0f}s")
for k, v in boot.items():
    if k.startswith("_"): continue
    print(k, {m: (round(v[m]["delta"], 4), round(v[m]["ci_lo"], 4), round(v[m]["ci_hi"], 4), v[m]["excludes_zero"]) for m in KEYS})
(EXP_DRIVE / f"{TODAY}-bootstrap-{PROTOCOL}-{HEAD}.json").write_text(json.dumps(boot, indent=2))

{'A0': 3, 'A1': 3, 'A2': 3, 'A8': 3} | GPU: True
[bootstrap] 100/1000  56s, còn ~507s
[bootstrap] 200/1000  113s, còn ~451s
[bootstrap] 300/1000  169s, còn ~394s
[bootstrap] 400/1000  225s, còn ~338s
[bootstrap] 500/1000  282s, còn ~282s
[bootstrap] 600/1000  338s, còn ~225s
[bootstrap] 700/1000  394s, còn ~169s
[bootstrap] 800/1000  451s, còn ~113s
[bootstrap] 900/1000  507s, còn ~56s
[bootstrap] 1000/1000  563s, còn ~0s
xong 570s
A8-A0 {'AP_vt': (0.0071, -0.0002, 0.0114, False), 'AP_t': (0.0042, 0.0005, 0.0072, True), 'AP': (0.0025, 0.0006, 0.0045, True)}
A2-A0 {'AP_vt': (0.009, 0.0051, 0.0153, True), 'AP_t': (0.0062, 0.0043, 0.0084, True), 'AP': (0.0035, 0.0018, 0.0057, True)}
A2-A8 {'AP_vt': (0.0019, -0.003, 0.0124, False), 'AP_t': (0.002, -0.001, 0.0057, False), 'AP': (0.0011, -0.001, 0.003, False)}
A1-A0 {'AP_vt': (0.0074, 0.0033, 0.0161, True), 'AP_t': (0.0016, -0.0005, 0.0032, False), 'AP': (0.0006, -0.001, 0.0024, False)}


3548